# Membuat Data Splits json


In [2]:
import os
import shutil
import pandas as pd
import numpy as np
import json
from google.colab import drive

# 1. Sambungkan Google Drive ke Colab
drive.mount('/content/drive')

# 2. Buat folder data/raw sesuai Kontrak 1
os.makedirs('/content/data/raw', exist_ok=True)
os.makedirs('/content/data/hasil', exist_ok=True) # Sekalian buat folder hasil

# 3. Tentukan path file di Google Drive
path_drive_fd002 = '/content/drive/MyDrive/Praktikum MLN/Dataset ML/train_FD002.txt'
path_drive_fd004 = '/content/drive/MyDrive/Praktikum MLN/Dataset ML/train_FD004.txt'

# Copy file dari Drive ke folder lokal Colab (data/raw/)
shutil.copy(path_drive_fd002, '/content/data/raw/train_FD002.txt')
shutil.copy(path_drive_fd004, '/content/data/raw/train_FD004.txt')
print("File berhasil disalin dari Drive ke data/raw/")

# ==========================================
# PROSES PEMBUATAN SPLITS.JSON
# ==========================================

def get_unit_ids(filepath, prefix):
    # Baca CSV tanpa header, dipisah spasi
    df = pd.read_csv(filepath, sep=r'\s+', header=None)
    units = df[0].unique()
    return [f"{prefix}_{int(u)}" for u in units]

units_fd002 = get_unit_ids('/content/data/raw/train_FD002.txt', 'FD002')
units_fd004 = get_unit_ids('/content/data/raw/train_FD004.txt', 'FD004')

def split_60_20_20(unit_list, seed=42):
    np.random.seed(seed)
    shuffled = np.random.permutation(unit_list)
    n = len(shuffled)

    idx_a = int(0.6 * n)
    idx_b = int(0.8 * n)

    return {
        'A': shuffled[:idx_a].tolist(),
        'B': shuffled[idx_a:idx_b].tolist(),
        'C': shuffled[idx_b:].tolist()
    }

split_002 = split_60_20_20(units_fd002, seed=42)
split_004 = split_60_20_20(units_fd004, seed=42)

splits_final = {
    "seed": 42,
    "A": split_002['A'] + split_004['A'],
    "B": split_002['B'] + split_004['B'],
    "C": split_002['C'] + split_004['C']
}

# Simpan ke data/splits.json
with open('/content/data/splits.json', 'w') as f:
    json.dump(splits_final, f)

print("\nFile splits.json berhasil dibuat di folder /content/data/ !")
print(f"Jumlah Mesin -> Kelompok A: {len(splits_final['A'])}, B: {len(splits_final['B'])}, C: {len(splits_final['C'])}")

Mounted at /content/drive
File berhasil disalin dari Drive ke data/raw/

File splits.json berhasil dibuat di folder /content/data/ !
Jumlah Mesin -> Kelompok A: 305, B: 102, C: 102
